In [43]:
using LowLevelFEM, LinearAlgebra

In [44]:
Threads.nthreads()
LinearAlgebra.BLAS.get_num_threads()

2

In [45]:
structured_box_mesh(n=20, order=2)

mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=3, field=:u)

Problem("structured_box", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 68921, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false)

In [46]:
prob = Problem([mat])
@time K1 = stiffnessMatrix(prob)

 15.507809 seconds (39.34 M allocations: 50.155 GiB, 19.25% gc time)


sparse([1, 2, 3, 79, 80, 81, 139, 140, 141, 142  …  206631, 206641, 206642, 206643, 206755, 206756, 206757, 206761, 206762, 206763], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763], [877.4928774928592, 320.51282051281675, -320.51282051281373, -156.69515669515715, -80.12820512820473, -21.367521367517345, 176.6381766381952, 160.2564102564159, 85.47008547007493, -156.69515669515195  …  -410.25641025639266, -3.595346242946107e-12, -7.926104217403918e-12, 364.67236467241906, -3.105782298007398e-11, -3.8120617773529375e-12, 364.67236467240537, -1.7024603948812e-11, -2.2389201603800757e-11, 32091.168091167936], 206763, 206763)

In [47]:
μ = mat.μ
λ = mat.λ
D = [λ+2μ λ λ 0 0 0; λ λ+2μ λ 0 0 0; λ λ λ+2μ 0 0 0; 0 0 0 μ 0 0; 0 0 0 0 μ 0; 0 0 0 0 0 μ]

6×6 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0      0.0      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0      0.0      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0      0.0      0.0
 0.0        0.0        0.0        76923.1      0.0      0.0
 0.0        0.0        0.0            0.0  76923.1      0.0
 0.0        0.0        0.0            0.0      0.0  76923.1

In [48]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), assembly=:csc, threads=1);

  4.063989 seconds (649.86 k allocations: 812.672 MiB, 0.91% gc time)


In [49]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), assembly=:csc);

  3.221335 seconds (650.20 k allocations: 1.633 GiB, 7.11% gc time)


In [50]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), assembly=:ijv, threads=1);

  7.664938 seconds (908.36 k allocations: 11.947 GiB, 3.49% gc time)


In [51]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), assembly=:ijv, threads=2);

  5.128212 seconds (908.56 k allocations: 11.947 GiB, 9.52% gc time)


In [52]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), assembly=:ijv);

  8.490774 seconds (908.89 k allocations: 11.948 GiB, 6.38% gc time)


In [53]:
norm(K1.A - K2.A) / norm(K1.A)

2.390470855094232e-16

In [54]:
using Profile

Kpattern = build_csc_pattern(Pu, Pu; Ω="body")

# Compilation
fill!(Kpattern.nzval, 0.0)
∫(
    SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu);
    Ω="body",
    assembly=:csc,
    threads=1,
    csc_matrix=Kpattern
)

fill!(Kpattern.nzval, 0.0)
Profile.Allocs.clear()

Profile.Allocs.@profile sample_rate=0.0001 begin
    ∫(
        SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu);
        Ω="body",
        assembly=:csc,
        threads=1,
        csc_matrix=Kpattern
    )
end

prof = Profile.Allocs.fetch()
length(prof.allocs)

0

In [93]:
structured_rect_mesh(x0=10.0, n=100, order=2)

In [118]:
prob = Problem([mat], type=:AxiSymmetric)

@time K1 = stiffnessMatrix(prob)

  0.590505 seconds (4.44 M allocations: 1.076 GiB, 37.35% gc time)


sparse([1, 2, 9, 10, 207, 208, 1399, 1400, 1599, 1600  …  1003, 1004, 21201, 21202, 80401, 80402, 80797, 80798, 80801, 80802], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802], [6.766856620432596e6, 3.0206816132549527e6, 376104.95304940705, -201464.69811888237, -5.264342493157104e6, 805697.6851598419, -1.1009134905033892e6, 201249.88836457385, 912807.3491419425, -804999.5534586578  …  -2148.0975400363095, -2.4563065760865193e7, -5.902972042435417e6, -4.249581365189443e6, 8.085626177489758e-7, -946022.1570238043, 2148.097537484835, -2.456306576086626e7, -1.6426201909780502e-6, 6.802079749162653e7], 80802, 80802)

In [100]:
Pu = Problem([mat], type=:VectorField, dim=2, field=:u)

E = mat.E
ν = mat.ν

r = ScalarField(Pu, "body", (x, y, z)->x)
A1 = [1 0 0; 0 0 0; 0 1 0; 0 0 1]
A2 = [0 0; 1/r 0; 0 0; 0 0]
B = A1 ⋅ SymGrad(Pu) + A2 ⋅ Pu
D = E / (1+ν) / (1-2ν) * [1-ν ν ν 0; ν 1-ν ν 0; ν ν 1-ν 0; 0 0 0 (1-2ν)/2]

4×4 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0
 0.0        0.0        0.0        76923.1

In [104]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), assembly=:csc, threads=1);

  1.931253 seconds (13.01 M allocations: 834.711 MiB, 5.27% gc time)


In [123]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), assembly=:csc);

  1.304769 seconds (13.01 M allocations: 893.589 MiB, 9.12% gc time)


In [109]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), assembly=:ijv, threads=1);

  2.083725 seconds (14.53 M allocations: 1.190 GiB, 5.58% gc time)


In [110]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), assembly=:ijv, threads=2);

  1.808168 seconds (14.53 M allocations: 1.190 GiB, 20.29% gc time)


In [111]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), assembly=:ijv);

  2.008810 seconds (14.53 M allocations: 1.190 GiB, 17.18% gc time)


In [82]:
norm(K1.A - K2.A) / norm(K1.A)

3.6544389404335964e-14